# CSMODEL MP Phase 1

In [1]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt

# Importing

In [2]:
# Defining data for the dataframe
LFS_df = pd.read_csv('LFSPUFApril2016.CSV', dtype=pd.StringDtype())
LFS_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180862 entries, 0 to 180861
Data columns (total 50 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   PUFREG           180862 non-null  string
 1   PUFPRV           180862 non-null  string
 2   PUFPRRCD         180862 non-null  string
 3   PUFHHNUM         180862 non-null  string
 4   PUFURB2K10       180862 non-null  string
 5   PUFPWGTFIN       180862 non-null  string
 6   PUFSVYMO         180862 non-null  string
 7   PUFSVYYR         180862 non-null  string
 8   PUFPSU           180862 non-null  string
 9   PUFRPL           180862 non-null  string
 10  PUFHHSIZE        180862 non-null  string
 11  PUFC01_LNO       180862 non-null  string
 12  PUFC03_REL       180862 non-null  string
 13  PUFC04_SEX       180862 non-null  string
 14  PUFC05_AGE       180862 non-null  string
 15  PUFC06_MSTAT     180862 non-null  string
 16  PUFC07_GRADE     180862 non-null  string
 17  PUFC08_CUR

# Data Cleaning

In [ ]:
LFS_df

In [ ]:
# LFS_df['PUFC10_CONWR'].unique()
LFS_df.groupby('PUFC05_AGE')['PUFC10_CONWR'].value_counts()
LFS_df['PUFC10_CONWR'] = LFS_df['PUFC10_CONWR'].replace({' ':'below working age',
                                                         '1':'OCW',
                                                         '2':'not OCW worker',
                                                         '3':'diplomatic',
                                                         '4':'student or tourist',
                                                         '5':'other'}) # one space
# LFS_df['PUFC10_CONWR'].unique()
LFS_df['PUFC14_PROCC'] = LFS_df['PUFC14_PROCC'].replace({'  ':'-1'}) # two spaces
# LFS_df['PUFC14_PROCC'].unique()

LFS_df['PUFC16_PKB'] = LFS_df['PUFC16_PKB'].replace({'  ':'-1'}) # two spaces
LFS_df['PUFC16_PKB'].unique()

## `PUFC24_PBASIS`, `PUFC25_PBASIC`, and `PUFNEWEMPSTAT`

- Pandas `Categorical` type might be more performant than using strings directly
- `Categorical.from_codes()` treats values in the column as category indices starting at zero
    - `-1` will get assigned `NaN` instead of a category

In [3]:
PUFC24_PBASIS_categories = pd.CategoricalDtype([
    "In Kind only",
    "Per piece",
    "Per Hour",
    "Per Day",
    "Monthly",
    "Pakyaw",
    "Other Salaries/Wages",
    "Commission Basis"])

# Replace blanks with -1 so that Categorical.from_codes() will replace them with NaNs
PUFC24_PBASIS_codes = LFS_df['PUFC24_PBASIS'].replace({' ':'-1'}).astype(int)

PUFC24_PBASIS = pd.Categorical.from_codes(
    codes=PUFC24_PBASIS_codes, 
    dtype=PUFC24_PBASIS_categories)

LFS_df['PUFC24_PBASIS'] = pd.Series(PUFC24_PBASIS)

- **TODO**: Should values be binned into categories in the Excel sheet?

In [ ]:
# pd.to_numeric() with errors='coerce' will coerce parse errors (i.e., blanks) into NaN 
LFS_df['PUFC25_PBASIC'] = pd.to_numeric(LFS_df['PUFC25_PBASIC'], errors='coerce')

In [5]:
PUFNEWEMPSTAT_categories = pd.CategoricalDtype([
    "Employed",
    "Unemployed",
    "Not in the labor force"])

# For some weird reason this column uses indices of 1-3 instead of 0-2
# So we replace blanks with 0 then subtract 1 from all values, giving -1 for blanks and an index range of 0-2
PUFNEWEMPSTAT_codes = LFS_df['PUFNEWEMPSTAT'].replace({' ':'0'}).astype(int) - 1

PUFNEWEMPSTAT = pd.Categorical.from_codes(
    codes=PUFNEWEMPSTAT_codes, 
    dtype=PUFNEWEMPSTAT_categories)

LFS_df['PUFNEWEMPSTAT'] = pd.Series(PUFNEWEMPSTAT)

In [6]:
# Inspect df
LFS_df[['PUFC24_PBASIS', 'PUFC25_PBASIC', 'PUFNEWEMPSTAT']]

,PUFC24_PBASIS,PUFC25_PBASIC,PUFNEWEMPSTAT
0,NaN,<NA>,Employed
1,NaN,<NA>,Employed
2,Per Day,250,Employed
3,NaN,<NA>,Employed
4,Monthly,115,Employed
...,...,...,...
180857,NaN,<NA>,Employed
180858,NaN,<NA>,Not in the labor force
180859,NaN,<NA>,NaN
180860,NaN,<NA>,NaN
